# Sequence Footprint Plotting

For each high-impact day identified in `01_dataset_analysis.ipynb`, this
notebook constructs a 5-day sequence plot showing:

- **Top row:** Cold-Dry (CD) compound event footprint  
- **Bottom row:** Cold-Wet (CW) compound event footprint  

Incident locations are overlaid as scaled scatter points.

**Inputs**
| File | Description |
|------|-------------|
| `output/top_days_pfpi_dictionaries/footprint_data.pkl` | High-impact day metadata (from notebook 1) |
| `data/arrays/{month}_min_temp_all.npy` | Monthly minimum temperature arrays |
| `data/arrays/{month}_precip_all.npy` | Monthly precipitation arrays |
| `data/percentiles/minT15P_30yr_filtered.npy` | 15th-percentile temperature threshold |
| `data/percentiles/perc15P_30yr_filtered.npy` | 15th-percentile precipitation threshold |
| `data/percentiles/perc85P_30yr_filtered.npy` | 85th-percentile precipitation threshold |
| `data/all_incident_data.csv` | Full incident dataset with lat/lon |
| `data/scotland_rail.shp` | Rail network shapefile |
| HadUK-Grid sample `.nc` | Used to read lat/lon coordinate grid |

**Configuration:** Edit `DATA_DIR`, `ARRAYS_DIR`, `PERCENTILES_DIR`, and
`HADUK_SAMPLE_FILE` in the cell below before running.


In [ ]:
import os
import pickle
import calendar
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
from netCDF4 import Dataset
from pathlib import Path
from datetime import timedelta

# ── User configuration ──────────────────────────────────────────────────────
# Root data directory
DATA_DIR = "data"

# Directory containing monthly temperature and precipitation arrays
ARRAYS_DIR = os.path.join(DATA_DIR, "arrays")

# Directory containing percentile threshold arrays
PERCENTILES_DIR = os.path.join(DATA_DIR, "percentiles")

# Path to a sample HadUK-Grid NetCDF file (any month/year, used for lat/lon only)
HADUK_SAMPLE_FILE = (
    "/path/to/HadUK-Grid/5km/tasmin/day/latest/"
    "tasmin_hadukgrid_uk_5km_day_19600101-19600131.nc"
)

# Full path to the incident CSV (with Lat, Long, Date, PfPI Minutes columns)
INCIDENT_CSV = os.path.join(DATA_DIR, "all_incident_data.csv")

# Path to the rail network shapefile
RAIL_SHAPEFILE = os.path.join(DATA_DIR, "scotland_rail.shp")

# Root output directory
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Footprint pickle produced by notebook 1 (choose count or PfPI version)
FOOTPRINT_PKL = os.path.join(OUTPUT_DIR, "top_days_pfpi_dictionaries", "footprint_data.pkl")

# Map extent [lon_min, lon_max, lat_min, lat_max]
MAP_EXTENT = [-8.0, -0.8, 54.5, 60.9]

# Fill value used to mark invalid cells in percentile arrays
FILL_VALUE = 500

print("Configuration complete.")


## 1. Core footprint function

In [ ]:
def make_footprint_core(input_month, year_index, day_index, date, WP):
    """
    Compute CD and CW compound-event classification for a single day
    and return the incident GeoDataFrame for that date.

    Parameters
    ----------
    input_month : str   e.g. 'feb'
    year_index  : int   Years since DATASET_START_YEAR (e.g. 61 for 2021)
    day_index   : int   Zero-based day-of-month index
    date        : str   ISO date string 'YYYY-MM-DD'
    WP          : str   Weather-pattern number (used for labelling only)

    Returns
    -------
    CD            : ndarray   Classification array (0–3)
    CW            : ndarray   Classification array (0–3)
    incident_counts : DataFrame  Incident counts per (Long, Lat) location
    rail          : GeoDataFrame Rail network geometry
    lon, lat      : ndarray   Coordinate grids
    """
    # Load monthly arrays
    temp_all   = np.load(os.path.join(ARRAYS_DIR, f"{input_month}_min_temp_all.npy"))
    precip_all = np.load(os.path.join(ARRAYS_DIR, f"{input_month}_precip_all.npy"))

    # Load percentile thresholds and mask fill values
    perc_minT_15 = np.load(os.path.join(PERCENTILES_DIR, "minT15P_30yr_filtered.npy"))
    perc_prec_15 = np.load(os.path.join(PERCENTILES_DIR, "perc15P_30yr_filtered.npy"))
    perc_prec_85 = np.load(os.path.join(PERCENTILES_DIR, "perc85P_30yr_filtered.npy"))
    for arr in [perc_minT_15, perc_prec_15, perc_prec_85]:
        arr[arr > FILL_VALUE] = np.nan

    # Load coordinate grid from a sample HadUK-Grid file
    with Dataset(HADUK_SAMPLE_FILE, mode='r') as ds:
        lat = ds.variables['latitude'][:]
        lon = ds.variables['longitude'][:]

    # Extract the specific day
    temp_day   = temp_all[year_index,   day_index, :, :]
    precip_day = precip_all[year_index, day_index, :, :]

    # Threshold flags
    temp_met      = temp_day   <= perc_minT_15
    precip_met_CD = precip_day <= perc_prec_15
    precip_met_CW = precip_day >= perc_prec_85

    # Encode: 0 = neither, 1 = cold only, 2 = dry/wet only, 3 = compound
    CD = np.zeros(temp_day.shape)
    CD[temp_met]                       = 1
    CD[precip_met_CD]                  = 2
    CD[temp_met & precip_met_CD]       = 3

    CW = np.zeros(temp_day.shape)
    CW[temp_met]                       = 1
    CW[precip_met_CW]                  = 2
    CW[temp_met & precip_met_CW]       = 3

    # Load incident data and filter to this date
    df = pd.read_csv(INCIDENT_CSV)
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df['Long'], df['Lat']),
        crs="EPSG:4326"
    )
    filtered_incidents = gdf[gdf['Date'] == str(date)]
    incident_counts = (filtered_incidents
                       .groupby(['Long', 'Lat'])
                       .size()
                       .reset_index(name='count'))

    # Load rail shapefile
    rail = gpd.read_file(RAIL_SHAPEFILE)
    if rail.crs != "EPSG:4326":
        rail = rail.to_crs("EPSG:4326")

    return CD, CW, incident_counts, rail, lon, lat


## 2. Build 5-day window dictionary

For each high-impact day, expand to include the four preceding days so that
the sequence leading up to the event can be plotted.

In [ ]:
# Load the high-impact day metadata saved by notebook 1
with open(FOOTPRINT_PKL, 'rb') as f:
    original_data = pickle.load(f)

date_lookup = {pd.to_datetime(k): v for k, v in original_data.items()}

expanded_data = {}

for date_str, details in original_data.items():
    current_date = pd.to_datetime(date_str)
    window_dict  = {}

    # Build entries for t-4, t-3, t-2, t-1, t (the event day)
    for days_before in range(4, -1, -1):
        target_date  = current_date - timedelta(days=days_before)
        target_str   = target_date.strftime('%Y-%m-%d')
        input_month  = target_date.strftime('%b').lower()
        year         = target_date.year
        year_index   = year - 1960
        day_index    = target_date.day - 1   # zero-based

        # Sanity check
        max_days = calendar.monthrange(year, target_date.month)[1]
        if day_index >= max_days:
            print(f"Warning: invalid day_index {day_index} for {target_str} "
                  f"(month has {max_days} days)")

        if target_date in date_lookup:
            entry = date_lookup[target_date].copy()
            entry['year_index'] = year_index
            entry['day_index']  = day_index
        else:
            entry = {
                'input_month': input_month,
                'year_index':  year_index,
                'day_index':   day_index,
                'date':        target_date,
                'WP':          None,   # fill in manually below if needed
            }

        window_dict[target_str] = entry

    expanded_data[date_str] = window_dict

expanded_pkl = os.path.join(OUTPUT_DIR, 'expanded_footprint_data.pkl')
with open(expanded_pkl, 'wb') as f:
    pickle.dump(expanded_data, f)
print(f"Saved expanded window data for {len(expanded_data)} events to {expanded_pkl}")


### 2a. Manually assign WP values for preceding days (if needed)

For days that are not themselves high-impact days, the WP number is not
automatically available from the dataset. Use the cell below to fill in
the WP sequence for a specific event window, then re-save the pickle.

Replace `event_date` and `wps` with the relevant date and WP sequence
(ordered oldest to most recent, i.e. t-4 → t).


In [ ]:
with open(expanded_pkl, 'rb') as f:
    expanded_data = pickle.load(f)

# ── Edit these for each event window as needed ──────────────────────────────
event_date = '2022-11-18'
wps        = ['22', '21', '29', '21', '05']   # 5 values: t-4, t-3, t-2, t-1, t
# ─────────────────────────────────────────────────────────────────────────────

for i, date_key in enumerate(sorted(expanded_data[event_date].keys())):
    expanded_data[event_date][date_key]['WP'] = wps[i]

with open(expanded_pkl, 'wb') as f:
    pickle.dump(expanded_data, f)
print(f"Updated WP values for event {event_date}")


## 3. Plot 5-day sequence footprint

In [ ]:
# ── Select the event to plot ─────────────────────────────────────────────────
event_date = '2021-02-11'   # change as needed
# ─────────────────────────────────────────────────────────────────────────────

with open(expanded_pkl, 'rb') as f:
    expanded_data = pickle.load(f)

footprint_data = expanded_data[event_date]

# Compute footprints for all 5 days
footprint_outputs = []
for date_str, info in footprint_data.items():
    if info['WP'] is None:
        print(f"Skipping {date_str} — WP not set. Use section 2a to assign WP values.")
        continue

    CD, CW, incidents, rail, lon, lat = make_footprint_core(
        input_month=info['input_month'],
        year_index=info['year_index'],
        day_index=info['day_index'],
        date=date_str,
        WP=info['WP'],
    )
    footprint_outputs.append({
        'date':      date_str,
        'WP':        info['WP'],
        'CD':        CD,
        'CW':        CW,
        'incidents': incidents,
        'rail':      rail,
        'lon':       lon,
        'lat':       lat,
    })

print(f"Computed footprints for {len(footprint_outputs)} days.")


In [ ]:
# ── Colourmap definitions ─────────────────────────────────────────────────────
cmap_cd = mpl.colors.ListedColormap(['white', 'gold', '#1dc3e2', 'magenta'])
cmap_cw = mpl.colors.ListedColormap(['white', 'gold', 'dodgerblue', 'seagreen'])
bounds  = [0, 1, 2, 3, 4]
norm    = mpl.colors.BoundaryNorm(bounds, cmap_cd.N)

# ── Figure layout ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(
    nrows=2, ncols=5, figsize=(12, 8),
    subplot_kw={'projection': ccrs.PlateCarree()}
)
plt.subplots_adjust(top=0.95, bottom=0.05, left=0.05, right=0.88,
                    wspace=0.02, hspace=0.2)

# ── Top row: Cold-Dry ─────────────────────────────────────────────────────────
for col, data in enumerate(footprint_outputs):
    ax = axes[0, col]
    ax.pcolormesh(data['lon'], data['lat'], data['CD'],
                  cmap=cmap_cd, norm=norm, shading='auto')
    # Hatch compound (value == 3) cells
    mask = (data['CD'] == 3).astype(float)
    ax.contourf(data['lon'], data['lat'], mask, levels=[0.5, 1.5],
                hatches=['///'], colors='none', transform=ccrs.PlateCarree())
    data['rail'].plot(ax=ax, color='black', linewidth=0.7, alpha=0.7,
                      transform=ccrs.PlateCarree())
    ax.scatter(data['incidents']['Long'], data['incidents']['Lat'],
               s=data['incidents']['count'] * 10, color='black', alpha=0.6,
               transform=ccrs.PlateCarree())
    ax.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.set_title(data['date'], fontsize=11)

# ── Bottom row: Cold-Wet ──────────────────────────────────────────────────────
for col, data in enumerate(footprint_outputs):
    ax = axes[1, col]
    ax.pcolormesh(data['lon'], data['lat'], data['CW'],
                  cmap=cmap_cw, norm=norm, shading='auto')
    mask = (data['CW'] == 3).astype(float)
    ax.contourf(data['lon'], data['lat'], mask, levels=[0.5, 1.5],
                hatches=['xxx'], colors='none', transform=ccrs.PlateCarree())
    data['rail'].plot(ax=ax, color='black', linewidth=0.7, alpha=0.7,
                      transform=ccrs.PlateCarree())
    ax.scatter(data['incidents']['Long'], data['incidents']['Lat'],
               s=data['incidents']['count'] * 10, color='black', alpha=0.6,
               transform=ccrs.PlateCarree())
    ax.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.set_title(data['date'], fontsize=11)

# ── Colorbars ─────────────────────────────────────────────────────────────────
for cax_pos, cmap, label, tick_labels in [
    ([0.92, 0.55, 0.015, 0.35], cmap_cd, "Cold-Dry (CD)",  ['Neither', 'Cold', 'Dry',  'Cold-Dry']),
    ([0.92, 0.10, 0.015, 0.35], cmap_cw, "Cold-Wet (CW)",  ['Neither', 'Cold', 'Wet',  'Cold-Wet']),
]:
    cbar_ax = fig.add_axes(cax_pos)
    cbar    = fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), cax=cbar_ax)
    cbar.ax.set_yticks([0.5, 1.5, 2.5, 3.5])
    cbar.ax.set_yticklabels(tick_labels)
    cbar.set_label(label, rotation=270, labelpad=15)

# fig.savefig(os.path.join(OUTPUT_DIR, f"sequence_footprint_{event_date}.png"), dpi=300)
plt.show()


### 3a. Save event-day data for downstream use

Saves a clean pickle of CD/CW arrays and incident locations for the event day
only (without the GeoDataFrame rail object, which cannot be pickled on all
platforms).


In [ ]:
event_day_data = [
    {k: v for k, v in item.items() if k != 'rail'}
    for item in footprint_outputs
    if item['date'] == event_date
]

safe_name = event_date.replace('-', '')
out_path  = os.path.join(OUTPUT_DIR, f"obs_data_{safe_name}_clean.pkl")
with open(out_path, 'wb') as f:
    pickle.dump(event_day_data, f)
print(f"Saved event-day data to {out_path}")
